# HAM10000 期末實驗 03：E3b 固定回合敏感度分析（事後追加）

需要 **T4 GPU**（執行階段 → 變更執行階段類型）。必須先跑完 02，這份會檢查切分與 02 完全相同。

**為什麼要追加**
02 的 E3b 中，模型 B、C 用較小的驗證集（約 880 張）做提前停止，兩者停止的回合差異很大（例如種子 42 的 B 停在第 4 回合、C 停在第 14 回合）。
C − B 因此可能同時受到「是否有資料洩漏」和「訓練回合不同」兩個因素影響。

**這份做什麼**
- 模型 B、C 都**固定訓練 20 回合，不提前停止**，使用第 20 回合的模型評估。5 個種子 × 2 個模型，共 10 次訓練，約 2 小時。
- 其他設定（切分、類別權重＋資料增強、學習率、cosine 排程）與 02 完全相同。
- 每回合仍計算驗證集分數，只作紀錄，不用來選模型。測試集只在第 20 回合評估一次。
- 結果存在 `results_e3b_fixed_epochs`，**不會改動 02 的任何結果檔**。

**報告定位**：這是看到 02 結果後才追加的敏感度分析，正式結果仍以 02 為準，不得替換。

**可續跑**：每完成一次訓練就存檔，斷線後重新「全部執行」會略過已完成的部分。

## 0. 環境設定

In [ ]:
import sys, subprocess, os
IN_COLAB = 'google.colab' in sys.modules
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'], check=False)

USE_DRIVE = True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
BASE = os.environ.get('HAM_BASE') or ('/content/drive/MyDrive/ham10000_project' if (IN_COLAB and USE_DRIVE) else './ham10000_project')

In [ ]:
import time, json, random, platform, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from joblib import Parallel, delayed

import torch
import torch.nn as nn
import torchvision
from torchvision import models
from torchvision.transforms import v2 as T
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_recall_fscore_support,
                             average_precision_score, confusion_matrix, ConfusionMatrixDisplay)

# ===== 實驗設定（報告表 3-2、表 3-3、表 3-5）=====
SEEDS = [42, 43, 44, 45, 46]
PILOT_SEED = 42
PILOT_EPOCHS, DEFAULT_EPOCHS = 20, 12
PILOT_MIN_GAIN = 0.01        # 第 12 回合後驗證集 macro-F1 需提升超過此值才改用 20 回合
IMG_SIZE = 224
PATIENCE = 3
BATCH_SIZE = 64
LR = 3e-4
WEIGHT_DECAY = 1e-4
PRETRAINED = os.environ.get('HAM_SMOKE') != '1'   # 程式檢查時不下載預訓練權重
NUM_WORKERS = int(os.environ.get('HAM_WORKERS', 2))   # Windows 本機請設 0
VAL_RATIO, TEST_RATIO = 0.15, 0.15
RUN_E4 = False     # 這份不需要外部資料
SMOKE_TEST = os.environ.get('HAM_SMOKE') == '1'     # 只供程式檢查用，正式執行請保持 False
if SMOKE_TEST:
    IMG_SIZE, PILOT_EPOCHS, DEFAULT_EPOCHS, BATCH_SIZE = 64, 3, 2, 16

RES_DIR = os.path.join(BASE, 'results_e3b_fixed_epochs')
FORMAL_DIR = os.path.join(BASE, 'results_final_cnn')     # 02 的正式結果，只讀取
PRED_DIR = os.path.join(RES_DIR, 'predictions')
MODEL_DIR = os.path.join(RES_DIR, 'models')
CACHE_DIR = os.path.join(BASE, 'cache')
CHECK_DIR = os.path.join(BASE, 'data_check')
BASELINE_DIR = os.path.join(BASE, 'results_final_baseline')
for d in [RES_DIR, PRED_DIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('裝置：', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else '')
if DEVICE == 'cpu' and not SMOKE_TEST:
    print('⚠ 沒有偵測到 GPU，請到「執行階段 → 變更執行階段類型」選 T4 GPU')

CLASS_ORDER = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLS2IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
MEL = CLS2IDX['mel']
SPLITS = ['lesion', 'image', 'image_noleak', 'image_ctrl']
SPLIT_LABEL = {'lesion': '依病灶切分', 'image': '依影像切分（模型 A）',
               'image_noleak': '移除與測試集同病灶的影像（模型 B）', 'image_ctrl': '移除等量無關病灶（模型 C）'}

## 1. 讀取資料

In [ ]:
DATA_DIR = os.environ.get('HAM_DATA_DIR')
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download('kmader/skin-cancer-mnist-ham10000')

meta_path, img_paths = None, {}
for root, _, files in os.walk(DATA_DIR):
    for f in files:
        if f == 'HAM10000_metadata.csv' and meta_path is None:
            meta_path = os.path.join(root, f)
        if f.lower().endswith('.jpg'):
            img_paths.setdefault(os.path.splitext(f)[0], os.path.join(root, f))
assert meta_path is not None, '找不到 HAM10000_metadata.csv'
df = pd.read_csv(meta_path)
df['path'] = df['image_id'].map(img_paths)
assert df['path'].notna().all(), '有影像找不到檔案'
df = df.sort_values('image_id').reset_index(drop=True)
print(f'影像 {len(df):,} 張，病灶 {df.lesion_id.nunique():,} 個')
SMOKE_MODE = SMOKE_TEST
# 00 以 pHash 候選加 SSIM ≥ 0.90 找到、並經人工確認為同一病灶但病灶編號不同的影像。
# 切分前把它們視為同一個病灶，避免同一病灶被分到訓練集與測試集。原始編號保留在 lesion_id_original
LESION_MERGE = [('HAM_0004919', 'HAM_0000140')]
df['lesion_id_original'] = df['lesion_id']
if not SMOKE_MODE:
    for keep_id, merge_id in LESION_MERGE:
        assert {keep_id, merge_id}.issubset(set(df['lesion_id'])), f'找不到病灶編號 {keep_id} 或 {merge_id}'
        df.loc[df['lesion_id'] == merge_id, 'lesion_id'] = keep_id
    conf = os.path.join(CHECK_DIR, 'ham_confirmed_same_photo.csv')
    assert os.path.exists(conf), '找不到 ham_confirmed_same_photo.csv，請先用正式版 00 執行資料檢查'
    c_ = pd.read_csv(conf)
    found = {frozenset(x) for x in zip(c_['lesion_a'], c_['lesion_b'])}
    assert found == {frozenset(x) for x in LESION_MERGE}, f'00 確認的重複病灶與 LESION_MERGE 不一致：{found}。請人工確認後更新 LESION_MERGE'
print('合併後病灶數：', df['lesion_id'].nunique())
y = df['dx'].map(CLS2IDX).values

# E4 外部資料：訓練時預測全部 1,705 張，排除名單在分析時才套用
ext = None
if RUN_E4:
    ext_file = os.path.join(CHECK_DIR, 'isic2018_external.csv')
    assert os.path.exists(ext_file), '找不到 isic2018_external.csv，請先執行 00 筆記本，或把 RUN_E4 改成 False'
    ext = pd.read_csv(ext_file)
    assert {'exclude', 'exclude_sensitivity'}.issubset(ext.columns), 'isic2018_external.csv 是舊版，請用正式版 00 重新執行'
    assert ext['image_id'].is_unique and (len(ext) == 1705 or SMOKE_TEST), 'ISIC 2018 影像數不是 1,705 張'
    y_ext = ext['dx'].map(CLS2IDX).values
    print('外部資料（全部）：', len(ext), '正式保留', int((~ext['exclude']).sum()), '張，敏感度分析保留', int((~ext['exclude_sensitivity']).sum()), '張')

## 2. 資料切分（與 Baseline 筆記本完全相同）

In [ ]:
holdout = VAL_RATIO + TEST_RATIO
test_share = TEST_RATIO / holdout
assert (df.groupby('lesion_id')['dx'].nunique() == 1).all(), '合併後有病灶對應到多種診斷'
LES_DX = df.groupby('lesion_id')['dx'].first()
LES_SIZE = df.groupby('lesion_id').size()          # 每個病灶在整個資料集的影像數

def split_by_image(seed):
    idx = np.arange(len(df))
    tr, tmp = train_test_split(idx, test_size=holdout, stratify=df['dx'], random_state=seed)
    va, te = train_test_split(tmp, test_size=test_share, stratify=df['dx'].values[tmp], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(te)

def split_by_lesion(seed):
    les = df.groupby('lesion_id')['dx'].first()
    tr_l, tmp_l = train_test_split(les.index.values, test_size=holdout, stratify=les.values, random_state=seed)
    va_l, te_l = train_test_split(tmp_l, test_size=test_share, stratify=les.loc[tmp_l].values, random_state=seed)
    lid = df['lesion_id'].values
    return (np.where(np.isin(lid, tr_l))[0], np.where(np.isin(lid, va_l))[0], np.where(np.isin(lid, te_l))[0])

def control_removal(seed, tr, va, te, removed_b):
    """模型 C：從訓練集移除與測試集無關的「整個病灶」，每個類別移除的影像數與模型 B 相同。
    優先抽取在整個資料集有多張影像的病灶（模型 B 移除的病灶都屬於這種），不足時再用單張影像的病灶補足。"""
    lid, dx = df['lesion_id'].values, df['dx'].values
    rng = np.random.default_rng(seed)
    test_les, val_les = set(lid[te]), set(lid[va])
    tr_count = pd.Series(lid[tr]).value_counts()              # 每個病灶在訓練集的影像數
    target = pd.Series(dx[removed_b]).value_counts()
    chosen, report = [], []
    for c in CLASS_ORDER:
        need = int(target.get(c, 0))
        if need == 0:
            continue
        cand = sorted(l for l in tr_count.index if l not in test_les and LES_DX[l] == c)
        multi = [l for l in cand if LES_SIZE[l] >= 2]
        single = [l for l in cand if LES_SIZE[l] < 2]
        got, got_multi = 0, 0
        for l in list(rng.permutation(multi)) + list(rng.permutation(single)):
            k = int(tr_count[l])
            if got + k <= need:
                chosen.append(l); got += k
                got_multi += k if LES_SIZE[l] >= 2 else 0
            if got == need:
                break
        b_les = set(lid[removed_b][dx[removed_b] == c]); c_les = set(l for l in chosen if LES_DX[l] == c)
        report.append({'種子': seed, '類別': c, '模型 B 移除影像': need, '模型 C 移除影像': got,
                       '模型 B 移除病灶數': len(b_les), '模型 C 移除病灶數': len(c_les),
                       '模型 C 來自多張影像病灶的影像比例': round(got_multi / got, 3) if got else np.nan,
                       '模型 B 移除病灶也在驗證集的數量': len(b_les & val_les), '模型 C 移除病灶也在驗證集的數量': len(c_les & val_les)})
    removed_c = tr[np.isin(lid[tr], chosen)]
    return np.setdiff1d(tr, removed_c), pd.DataFrame(report)

CONTROL_REPORT = []
def get_splits(seed):
    tr, va, te = split_by_image(seed)
    lid = df['lesion_id'].values
    tr_noleak = tr[~np.isin(lid[tr], lid[te])]
    tr_ctrl, rep = control_removal(seed, tr, va, te, np.setdiff1d(tr, tr_noleak))
    CONTROL_REPORT.append(rep)
    # 模型 B、C 選擇回合用的驗證集：移除與測試集、模型 B 訓練集或模型 C 訓練集同病灶的影像
    # 這樣驗證集的病灶不會出現在任一方的訓練集，兩個模型的回合選擇不受訓練與驗證重疊程度不同的影響
    va_clean = va[~(np.isin(lid[va], lid[te]) | np.isin(lid[va], lid[tr_noleak]) | np.isin(lid[va], lid[tr_ctrl]))]
    return {'lesion': split_by_lesion(seed), 'image': (tr, va, te),
            'image_noleak': (tr_noleak, va_clean, te), 'image_ctrl': (tr_ctrl, va_clean, te)}

def seen_mask(seed):
    # 依影像切分的測試集中，哪些影像的病灶也出現在（模型 A 的）訓練集。模型 C 保留了這些影像
    tr, _, te = SPLITS_BY_SEED[seed]['image']
    lid = df['lesion_id'].values
    return np.isin(lid[te], lid[tr])

SPLITS_BY_SEED = {s: get_splits(s) for s in SEEDS}

# 切分指紋：結果資料夾若存有不同切分（例如合併病灶前）產生的結果，直接停止，避免新舊結果混用
import hashlib
_h = hashlib.sha256(repr(LESION_MERGE).encode())
for s in SEEDS:
    for name in ['lesion', 'image', 'image_noleak', 'image_ctrl']:
        for part in SPLITS_BY_SEED[s][name]:
            _h.update(np.asarray(part, dtype=np.int64).tobytes())
SPLIT_FINGERPRINT = _h.hexdigest()[:16]
_fp = os.path.join(RES_DIR, 'split_fingerprint.txt')
if os.path.exists(_fp):
    assert open(_fp).read().strip() == SPLIT_FINGERPRINT, f'{RES_DIR} 內的結果來自不同的資料切分。請把舊資料夾改名備份後重新執行'
else:
    assert not any(f.endswith(('.csv', '.json')) for f in os.listdir(RES_DIR)), f'{RES_DIR} 已有結果但沒有切分指紋，可能是舊版產出。請把舊資料夾改名備份後重新執行'
    open(_fp, 'w').write(SPLIT_FINGERPRINT)
print('切分指紋：', SPLIT_FINGERPRINT)
_fp_formal = os.path.join(FORMAL_DIR, 'split_fingerprint.txt')
assert os.path.exists(_fp_formal), '找不到 02 的結果，請先完成 02'
assert open(_fp_formal).read().strip() == SPLIT_FINGERPRINT, '切分與 02 不同，請勿執行'
print('切分與 02 相同')
control_report = pd.concat(CONTROL_REPORT, ignore_index=True)

# 檢查：模型 B 訓練集與測試集沒有同病灶，模型 C 與 B 的類別張數相同
for s in SEEDS:
    lid = df['lesion_id'].values
    trb, _, te = SPLITS_BY_SEED[s]['image_noleak']
    trc = SPLITS_BY_SEED[s]['image_ctrl'][0]
    assert not np.isin(lid[trb], lid[te]).any(), f'種子 {s}：模型 B 仍有測試病灶'
    assert set(lid[np.setdiff1d(SPLITS_BY_SEED[s]['image'][0], trc)]).isdisjoint(set(lid[te])), f'種子 {s}：模型 C 移除到測試病灶'
    vab = SPLITS_BY_SEED[s]['image_noleak'][1]
    assert not np.isin(lid[vab], lid[te]).any(), f'種子 {s}：模型 B、C 的驗證集仍有測試病灶'
    assert not np.isin(lid[vab], lid[trb]).any() and not np.isin(lid[vab], lid[trc]).any(), f'種子 {s}：模型 B、C 的驗證集仍有訓練病灶'
    vc = pd.Series(df['dx'].values[vab]).value_counts().reindex(CLASS_ORDER).fillna(0).astype(int)
    print(f'種子 {s} 模型 B、C 驗證集各類影像數：', vc.to_dict())
    assert SMOKE_MODE or (vc >= 3).all(), f'種子 {s}：模型 B、C 的驗證集有類別少於 3 張，無法可靠選擇回合'
r = control_report
bad = r[r['模型 B 移除影像'] != r['模型 C 移除影像']]
# 模型 B 與 C 每個類別的移除張數必須完全相同，否則 C − B 的比較前提不成立，停止執行
assert bad.empty, '模型 C 無法完全配對，請勿開始 E3b 訓練：\n' + bad.to_string(index=False)

# 與 Baseline 的切分名單比對
ref = os.path.join(BASELINE_DIR, 'split_assignment.csv')
if os.path.exists(ref):
    a = pd.read_csv(ref, keep_default_na=False).set_index('image_id').loc[df['image_id']]
    for s in SEEDS:
        for name in ['lesion', 'image']:
            tr, va, te = SPLITS_BY_SEED[s][name]
            col = np.full(len(df), '', dtype=object); col[tr], col[va], col[te] = 'train', 'val', 'test'
            assert (a[f'seed{s}_{name}'].values == col).all(), f'種子 {s} {name} 切分與 Baseline 不一致'
        for name in ['image_noleak', 'image_ctrl']:
            removed = np.isin(np.arange(len(df)), np.setdiff1d(SPLITS_BY_SEED[s]['image'][0], SPLITS_BY_SEED[s][name][0]))
            assert (a[f'seed{s}_{name}_removed'].astype(str).values == removed.astype(str)).all(), f'種子 {s} {name} 與 Baseline 不一致'
    print('切分與 Baseline 一致')
else:
    print('未找到 Baseline 切分檔，略過比對（程式與種子相同，結果應一致）')

## 3. 影像預先縮放（224×224，存成快取）

In [ ]:
def load_resized(path, size=IMG_SIZE):
    return np.asarray(Image.open(path).convert('RGB').resize((size, size), Image.BILINEAR), dtype=np.uint8)

def cached_images(paths, ids, name):
    f, fi = os.path.join(CACHE_DIR, f'{name}.npy'), os.path.join(CACHE_DIR, f'{name}_ids.txt')
    if os.path.exists(f) and os.path.exists(fi) and open(fi).read().split() == list(ids):
        return np.load(f, mmap_mode='r')
    t0 = time.time()
    arr = np.stack(Parallel(n_jobs=-1)(delayed(load_resized)(p) for p in paths))
    np.save(f, arr); open(fi, 'w').write('\n'.join(ids))
    print(f'{name}：縮放完成，耗時 {time.time()-t0:.0f} 秒')
    return np.load(f, mmap_mode='r')

IMAGES = cached_images(df['path'], df['image_id'], f'ham_images_{IMG_SIZE}px')
EXT_IMAGES = cached_images([os.path.join(BASE, p) for p in ext['path']], ext['image_id'], f'isic2018_all_images_{IMG_SIZE}px') if ext is not None else None
print('HAM10000：', IMAGES.shape, '外部資料：', None if EXT_IMAGES is None else EXT_IMAGES.shape)

## 4. 資料集、增強、模型（與期中相同）

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_aug = T.Compose([
    T.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), antialias=True),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
    T.RandomRotation(30),
    T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
])
to_input = T.Compose([T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])

class SkinDataset(Dataset):
    def __init__(self, idx, augment=False, images=None, labels=None):
        self.idx, self.augment = np.asarray(idx), augment
        self.images = IMAGES if images is None else images
        self.labels = y if labels is None else labels
    def __len__(self):
        return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        img = torch.from_numpy(np.array(self.images[j])).permute(2, 0, 1)
        if self.augment:
            img = train_aug(img)
        return to_input(img), int(self.labels[j])

def build_model():
    weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if PRETRAINED else None
    m = models.efficientnet_b0(weights=weights)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASS_ORDER))
    return m.to(DEVICE)

@torch.no_grad()
def predict(model, idx, images=None, labels=None):
    model.eval()
    loader = DataLoader(SkinDataset(idx, images=images, labels=labels), batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS)
    out = []
    for xb, _ in loader:
        with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
            out.append(model(xb.to(DEVICE)).float().softmax(1).cpu())
    return torch.cat(out).numpy()

def evaluate(y_true, y_pred, labels=None):
    labels = list(range(len(CLASS_ORDER))) if labels is None else labels
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=labels, average='macro', zero_division=0)
    mp, mr, _, _ = precision_recall_fscore_support(y_true, y_pred, labels=[MEL], zero_division=0)
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Precision (macro)': p, 'Recall (macro)': r,
            'F1-score (macro)': f, 'Balanced Accuracy': r,
            'Melanoma Recall': mr[0] if (y_true == MEL).any() else np.nan,          # 沒有黑色素瘤時不計算
            'Melanoma Precision': mp[0],                                          # 沒有判為黑色素瘤時記為 0
            'Melanoma FN': int(((y_true == MEL) & (y_pred != MEL)).sum()),
            'Melanoma FP': int(((y_true != MEL) & (y_pred == MEL)).sum())}

## 5. 訓練函式

提出方法為「類別權重＋資料增強」。類別權重 = 訓練影像數 ／（類別數 × 該類影像數），只依訓練集計算。

In [ ]:
CONFIGS = {
    'none':   dict(class_weight=False, augment=False, oversample=False, label='不處理'),
    'cw':     dict(class_weight=True,  augment=False, oversample=False, label='類別權重'),
    'aug':    dict(class_weight=False, augment=True,  oversample=False, label='資料增強'),
    'cw+aug': dict(class_weight=True,  augment=True,  oversample=False, label='類別權重＋資料增強（提出方法）'),
    'os+aug': dict(class_weight=False, augment=True,  oversample=True,  label='過採樣＋資料增強'),
}
MAIN = 'cw+aug'
E2_ORDER = ['none', 'cw', 'aug', 'cw+aug', 'os+aug']
RUNS_CSV = os.path.join(RES_DIR, 'all_runs.csv')

def tag(cfg, split, seed):
    return f'{cfg.replace("+", "-")}__{split}__seed{seed}'

def train_run(cfg_name, split, seed, epochs, save_dir=PRED_DIR, save_tag=None, *, pilot_only=False):
    cfg = CONFIGS[cfg_name]
    set_seed(seed)
    tr, va, te = SPLITS_BY_SEED[seed][split]
    if SMOKE_TEST:
        tr, va, te = tr[:96], va[:64], te[:64]
    counts = np.bincount(y[tr], minlength=len(CLASS_ORDER))
    g = torch.Generator().manual_seed(seed)
    ds_tr = SkinDataset(tr, augment=cfg['augment'])
    if cfg['oversample']:
        sw = 1.0 / np.maximum(counts[y[tr]], 1)
        sampler = WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(tr), replacement=True, generator=g)
        loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, drop_last=True)
    else:
        loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=NUM_WORKERS, drop_last=True)
    weight = None
    if cfg['class_weight']:
        weight = torch.tensor(len(tr) / (len(CLASS_ORDER) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight)

    model = build_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))

    best_f1, best_state, wait, history = -1, None, 0, []
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sched.step()
        val_f1 = f1_score(y[va], predict(model, va).argmax(1), labels=list(range(len(CLASS_ORDER))), average='macro', zero_division=0)
        history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'val_macro_f1': val_f1})
        print(f'  [種子 {seed} | {cfg_name} | {split}] epoch {epoch:2d}  loss {np.mean(losses):.4f}  val macro-F1 {val_f1:.4f}')
        if val_f1 > best_f1:
            best_f1, wait = val_f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:                 # 試跑也使用相同的提前停止規則
                print('  提前停止'); break
    secs = time.time() - t0
    model.load_state_dict(best_state)
    if pilot_only:
        # 回合數試跑只使用訓練/驗證集，禁止產生測試集或外部集分數與指標。
        t = save_tag or 'pilot'
        pd.DataFrame(history).to_csv(os.path.join(save_dir, f'history__{t}.csv'), index=False)
        return {'試跑種子': seed, '最佳驗證 macro-F1': float(best_f1), '訓練秒數': secs}, history

    t = save_tag or tag(cfg_name, split, seed)
    probs = {'val': predict(model, va), 'test': predict(model, te),
             'val_image_id': df.image_id.iloc[va].to_numpy(dtype=str),
             'test_image_id': df.image_id.iloc[te].to_numpy(dtype=str)}
    if EXT_IMAGES is not None:
        probs['ext'] = predict(model, np.arange(len(ext)), images=EXT_IMAGES, labels=y_ext)
        probs['ext_image_id'] = ext.image_id.to_numpy(dtype=str)
    np.savez_compressed(os.path.join(save_dir, f'prob__{t}.npz'), **probs)
    pd.DataFrame(history).to_csv(os.path.join(save_dir, f'history__{t}.csv'), index=False)
    if True:                              # 所有設定都保存最佳回合的權重
        torch.save(best_state, os.path.join(MODEL_DIR, f'model__{t}.pt'))

    row = {'實驗設定': cfg_name, '說明': cfg['label'], '切分': split, '種子': seed, '回合上限': epochs,
           '最佳回合': int(np.argmax([h['val_macro_f1'] for h in history]) + 1),
           '實際回合': len(history), '驗證集 macro-F1': best_f1, '訓練影像數': len(tr)}
    row.update(evaluate(y[te], probs['test'].argmax(1)))
    row['訓練秒數'] = round(secs, 1)
    return row, history

## 6. 固定 20 回合訓練（模型 B、C）

In [ ]:
FIXED_EPOCHS = 2 if SMOKE_TEST else 20
RUNS_CSV = os.path.join(RES_DIR, 'fixed_runs.csv')

def train_fixed(split, seed, epochs=FIXED_EPOCHS):
    """與 02 的 train_run 相同，但不提前停止，使用最後一回合的模型。"""
    cfg = CONFIGS[MAIN]
    set_seed(seed)
    tr, va, te = SPLITS_BY_SEED[seed][split]
    if SMOKE_TEST:
        tr, va, te = tr[:96], va[:64], te[:64]
    counts = np.bincount(y[tr], minlength=len(CLASS_ORDER))
    g = torch.Generator().manual_seed(seed)
    loader = DataLoader(SkinDataset(tr, augment=cfg['augment']), batch_size=BATCH_SIZE, shuffle=True, generator=g, num_workers=NUM_WORKERS, drop_last=True)
    weight = torch.tensor(len(tr) / (len(CLASS_ORDER) * np.maximum(counts, 1)), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weight)
    model = build_model()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=(DEVICE == 'cuda'))
    history, t0 = [], time.time()
    for epoch in range(1, epochs + 1):
        model.train(); losses = []
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE, non_blocking=True), yb.to(DEVICE, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(DEVICE, enabled=(DEVICE == 'cuda')):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            losses.append(loss.item())
        sched.step()
        val_f1 = f1_score(y[va], predict(model, va).argmax(1), labels=list(range(len(CLASS_ORDER))), average='macro', zero_division=0)
        history.append({'epoch': epoch, 'train_loss': float(np.mean(losses)), 'val_macro_f1': val_f1})
        print(f'  [種子 {seed} | {split} | 固定回合] epoch {epoch:2d}  loss {np.mean(losses):.4f}  val macro-F1 {val_f1:.4f}')
    secs = time.time() - t0
    t = tag(MAIN, split, seed) + '__fixed'
    probs = {'test': predict(model, te), 'test_image_id': df.image_id.iloc[te].to_numpy(dtype=str)}
    pd.DataFrame(history).to_csv(os.path.join(PRED_DIR, f'history__{t}.csv'), index=False)
    row = {'切分': split, '種子': seed, '回合': epochs, '第 20 回合驗證集 macro-F1': history[-1]['val_macro_f1'],
           '驗證集最佳回合（僅供參考）': int(np.argmax([h['val_macro_f1'] for h in history]) + 1), '訓練影像數': len(tr)}
    row.update(evaluate(y[te], probs['test'].argmax(1)))
    row['訓練秒數'] = round(secs, 1)
    # 先寫紀錄，最後才寫預測檔。預測檔先存成暫存檔再改名，存在就代表這次訓練完整結束
    runs = pd.read_csv(RUNS_CSV) if os.path.exists(RUNS_CSV) else pd.DataFrame(columns=['切分', '種子'])
    runs = runs[~((runs['切分'] == split) & (runs['種子'] == seed))]
    pd.concat([runs, pd.DataFrame([row])], ignore_index=True).to_csv(RUNS_CSV, index=False, encoding='utf-8-sig')
    tmp = os.path.join(PRED_DIR, f'tmp__{t}.npz')
    np.savez_compressed(tmp, **probs)
    os.replace(tmp, fixed_path(split, seed))
    return row

def fixed_path(split, seed):
    return os.path.join(PRED_DIR, f'prob__{tag(MAIN, split, seed)}__fixed.npz')

def is_done(split, seed):
    # 預測檔、訓練紀錄、彙總列三者都在，且紀錄有完整回合數，才算完成
    t = tag(MAIN, split, seed) + '__fixed'
    hist = os.path.join(PRED_DIR, f'history__{t}.csv')
    if not (os.path.exists(fixed_path(split, seed)) and os.path.exists(hist) and os.path.exists(RUNS_CSV)):
        return False
    if len(pd.read_csv(hist)) != FIXED_EPOCHS:
        return False
    runs = pd.read_csv(RUNS_CSV)
    if not ((runs['切分'] == split) & (runs['種子'] == seed)).any():
        return False
    try:
        z = np.load(fixed_path(split, seed)); _ = z['test'].shape
    except Exception:
        return False
    return True

for seed in SEEDS:
    for split in ['image_noleak', 'image_ctrl']:
        if is_done(split, seed):
            print(f'已完成，略過：種子 {seed} {SPLIT_LABEL[split]}'); continue
        print(f'開始：種子 {seed} | {SPLIT_LABEL[split]}')
        row = train_fixed(split, seed)
        print(f'完成：test macro-F1 {row["F1-score (macro)"]:.3f}，耗時 {row["訓練秒數"]:.0f} 秒\n')
assert all(is_done(sp, s) for s in SEEDS for sp in ['image_noleak', 'image_ctrl']), '仍有訓練未完成，請重新執行這一格'
print('10 次訓練全部完成')

## 7. 分析：固定回合與 02 正式結果並列

- 每個種子分「全部」「病灶出現過」「病灶未出現過」三群，計算 macro-F1 的 C − B。
- **分群差距** =（病灶出現過的 C − B）−（病灶未出現過的 C − B），若資料洩漏是主因，應為正值。
- 02 正式結果（提前停止）一併列出，方便對照。只作描述，不做顯著性檢定。

In [ ]:
def load_test_pred(path, seed):
    z = np.load(path)
    te = SPLITS_BY_SEED[seed]['image'][2]
    if SMOKE_TEST:
        te = te[:64]
    assert 'test_image_id' in z.files or SMOKE_TEST, '預測檔沒有影像編號，可能是舊版 02 的結果'
    if 'test_image_id' in z.files:
        assert (z['test_image_id'] == df.image_id.iloc[te].to_numpy(dtype=str)).all(), '測試集影像編號對不上'
    return z['test'].argmax(1)

def formal_path(split, seed):
    return os.path.join(FORMAL_DIR, 'predictions', f'prob__{tag(MAIN, split, seed)}.npz')

ALL_LABELS = list(range(len(CLASS_ORDER)))

def f1m(yt, yp, labels):
    return f1_score(yt, yp, labels=labels, average='macro', zero_division=0)

rows = []
for s in SEEDS:
    te = SPLITS_BY_SEED[s]['image'][2]
    seen = seen_mask(s)
    if SMOKE_TEST:
        te, seen = te[:64], seen[:64]
    yt = y[te]
    common = sorted(set(yt[seen]) & set(yt[~seen]))
    for version, getp in [('固定 20 回合', fixed_path), ('02 正式（提前停止）', formal_path)]:
        pb, pc = load_test_pred(getp('image_noleak', s), s), load_test_pred(getp('image_ctrl', s), s)
        r = {'版本': version, '種子': s}
        for gname, mask, labels in [('全部', np.ones(len(te), bool), ALL_LABELS), ('病灶出現過', seen, common), ('病灶未出現過', ~seen, common)]:
            b, c = f1m(yt[mask], pb[mask], labels), f1m(yt[mask], pc[mask], labels)
            r[f'{gname} B'], r[f'{gname} C'], r[f'{gname} C−B'] = b, c, c - b
        r['分群差距'] = r['病灶出現過 C−B'] - r['病灶未出現過 C−B']
        rows.append(r)
per_seed = pd.DataFrame(rows)
per_seed.to_csv(os.path.join(RES_DIR, 'E3b_fixed_per_seed.csv'), index=False, encoding='utf-8-sig')

# 檢查：重新計算的 02 正式結果必須與 02 輸出的 E3b_cnn_per_seed.csv 相同
_ref = os.path.join(FORMAL_DIR, 'E3b_cnn_per_seed.csv')
if os.path.exists(_ref) and not SMOKE_TEST:
    r_ = pd.read_csv(_ref)
    r_ = r_[(r_['指標'] == 'F1-score (macro)')].set_index(['種子', '群組'])
    f_ = per_seed[per_seed['版本'].str.startswith('02')].set_index('種子')
    for s in SEEDS:
        for gname in ['全部', '病灶出現過', '病灶未出現過']:
            assert abs(r_.loc[(s, gname), 'C'] - r_.loc[(s, gname), 'B'] - f_.loc[s, f'{gname} C−B']) < 1e-6, f'種子 {s} {gname} 與 02 結果不一致'
    print('02 正式結果重算一致')

def fmt(v):
    v = pd.Series(v); return f'{v.mean():.3f} ± {v.std(ddof=1):.3f}'
summary = []
for version, d in per_seed.groupby('版本', sort=False):
    for col in ['全部 B', '全部 C', '全部 C−B', '病灶出現過 C−B', '病灶未出現過 C−B', '分群差距']:
        row = {'版本': version, '項目': col, '平均 ± 標準差': fmt(d[col])}
        if 'C−B' in col or col == '分群差距':
            row['為正的種子數'] = f'{int((d[col] > 0).sum())}/{len(d)}'
            row['各種子'] = '、'.join(f'{v:+.3f}' for v in d[col])
        summary.append(row)
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(RES_DIR, 'E3b_fixed_summary.csv'), index=False, encoding='utf-8-sig')
print(summary.to_string(index=False))

In [ ]:
# 圖：兩個版本的分群 C − B（每個點是一個種子）
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, (version, d) in zip(axes, per_seed.groupby('版本', sort=False)):
    for s_, r in d.iterrows():
        ax.plot([0, 1], [r['病灶出現過 C−B'], r['病灶未出現過 C−B']], marker='o', color='gray', alpha=0.7)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_xticks([0, 1]); ax.set_xticklabels(['Lesion seen', 'Lesion not seen'])
    ax.set_title('Fixed 20 epochs' if version.startswith('固定') else 'Early stopping (formal)')
axes[0].set_ylabel('Macro F1 difference (C − B)')
plt.tight_layout(); plt.savefig(os.path.join(RES_DIR, 'fig_E3b_fixed_vs_formal.png'), dpi=300); plt.show()

## 8. 打包

執行完把雲端硬碟的 `results_e3b_fixed_epochs` 資料夾（不含 predictions）交給組長，或直接下載下面的 zip。

In [ ]:
out = os.path.join(BASE, 'results_e3b_fixed_epochs_for_report')
os.makedirs(out, exist_ok=True)
for f in os.listdir(RES_DIR):
    if f.endswith(('.csv', '.png', '.txt')):
        shutil.copy(os.path.join(RES_DIR, f), out)
shutil.make_archive(out, 'zip', out)
print('已打包：', out + '.zip')